<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/01_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Data Pipeline

Загрузка и исследование данных, фиксированный train/validation split, resize с padding, преобразование в тензоры, Dataset/DataLoader и проверка батчей.

> Этот notebook самодостаточен: его можно запускать сверху вниз в чистом Google Colab. Он не требует выполнения других notebook-файлов проекта.

In [ ]:
import os

# Должно быть установлено до первого использования CUDA
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# 1. Загрузка и осмотр данных

In [ ]:
from torchvision.datasets import OxfordIIITPet
import matplotlib.pyplot as plt
import random

dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

print("Количество изображений:", len(dataset))

image, label = dataset[0]

print("Тип изображения:", type(image))
print("Размер (ширина, высота):", image.size)
print("Цветной режим:", image.mode)
print("Метка класса:", label)

indices = random.Random(42).sample(range(len(dataset)), 12)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, index in zip(axes.flat, indices):
  image, label = dataset[index]

  ax.imshow(image)
  ax.set_title(f"Class {label} | {image.size}")
  ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

rows = []

for index in range(len(dataset)):
  image, label = dataset[index]

  size = image.size
  rows.append({
      'index': index,
      'label': label,
      'width': size[0],
      'height': size[1],
      'aspect_ratio': size[0] / size[1]
  })

metadata = pd.DataFrame(rows)


print(metadata.head())

print(
    metadata[["width", "height", "aspect_ratio"]].describe()
)

class_counts = metadata["label"].value_counts().sort_index()

print("Количество классов:", len(class_counts))
print("Минимум изображений на класс:", class_counts.min())
print("Максимум изображений на класс:", class_counts.max())

## Фиксация train и validation

Разделим 3680 изображений из `trainval` в пропорции 80/20. Используем стратификацию, чтобы сохранить примерно одинаковые доли пород в обеих частях:

In [ ]:
from sklearn.model_selection import train_test_split

train_indices, val_indices = train_test_split(
    metadata["index"].to_numpy(),
    test_size=0.2,
    random_state=42,
    stratify=metadata["label"].to_numpy(),
)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Пересечение:", len(set(train_indices) & set(val_indices)))

## Выбор способа изменения размера

| Подход | Преимущество | Недостаток |
| :--- | :--- | :--- |
| Растянуть до квадрата | Простота | Искажаются пропорции |
| Масштабировать и обрезать | Сохраняются пропорции | Можно отрезать часть животного |
| Масштабировать и добавить поля | Сохраняются пропорции и весь кадр | Часть площади занимают поля |


Для первой CNN будем использовать **масштабирование с добавлением полей до `224×224`** это исходное решение, качество которого позже можно будет сравнить с другими вариантами

Например, фотография размером 400 × 200:
  1. Уменьшается до 224 × 112
  2. Получает по **56 пикселей поля сверху и снизу**
  3. Итоговый размер - 224 × 224

Важно: пропорции сохраняются, но мелкие детали при уменьшении могут потеряться


### Преобразование размера:


In [ ]:
from torchvision import transforms
from torchvision.transforms import functional as TF


class ResizeWithPadding:
    def __init__(self, size=224):
        self.size = size

    def __call__(self, image):
        image = image.convert("RGB")
        width, height = image.size

        scale = self.size / max(width, height)

        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [new_height, new_width],
            antialias=True,
        )

        # Берём фактические размеры после resize
        width, height = image.size

        left = (self.size - width) // 2
        right = self.size - width - left

        top = (self.size - height) // 2
        bottom = self.size - height - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=0,
        )

`__call__` позволяет использовать экземляр класса как функцию:

```
resize = ResizeWithPadding(224)
resized_image = resize(image)
```

Проверка


In [ ]:
image, label = dataset[int(train_indices[0])]

resize = ResizeWithPadding(224)
prepared_image = resize(image)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(image)
axes[0].set_title(f"Original: {image.size}")

axes[1].imshow(prepared_image)
axes[1].set_title(f"Prepared: {prepared_image.size}")

for ax in axes:
  ax.axis("off")

plt.tight_layout()
plt.show()

Добавление преобразования в тензор:

In [ ]:
basic_transform = transforms.Compose([
    ResizeWithPadding(224),
    transforms.ToTensor(),
])

image_tensor = basic_transform(image)

print("Shape:", image_tensor.shape)
print("Dtype:", image_tensor.dtype)
print("Range:", image_tensor.min().item(), image_tensor.max().item())

### Создаем обучающую и валидационную части

In [ ]:
from torch.utils.data import Subset, DataLoader

train_source = OxfordIIITPet(
    root='data',
    split='trainval',
    target_types='category',
    transform=basic_transform,
    download=False,
)

val_source = OxfordIIITPet(
    root='data',
    split='trainval',
    target_types='category',
    transform=basic_transform,
    download=False,
)

train_dataset = Subset(train_source, train_indices.tolist())
val_dataset = Subset(val_source, val_indices.tolist())

train_source.transform = basic_transform
val_source.transform = basic_transform

class_names = train_source.classes

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Classes:", len(class_names))

### Собираем батчи

In [ ]:
import torch

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))

print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

### Самостоятельная часть


Напиши код, который:

* Получает первый батч из val_loader.
* Выводит первое изображение этого батча.
* Показывает название его породы в заголовке.

In [ ]:
first_batch = iter(val_loader)
val_image, val_label = next(first_batch)


print(val_image.shape, val_label.shape)

image_for_plot = val_image[0].permute(1, 2, 0)
breed_name = class_names[val_label[0].item()]

plt.figure(figsize=(5, 5))

plt.imshow(image_for_plot)
plt.title(breed_name)

plt.axis("off")
plt.tight_layout()
plt.show()